# Cascade Quantum — Full Pipeline Example

This notebook demonstrates the complete QML workflow using the Cascade Quantum SDK:
1. Connect to the server
2. Select a dataset
3. Run PCA analysis
4. Run Mutual Information analysis
5. Run Quantum Resource Estimation
6. Train a model
7. Run inference (classification)
8. Run Explainable AI

All computation happens on the server. This notebook only sends requests and displays results.

In [1]:
import cascade_quantum as cq

# Connect to the server
# Reads CQ_URL and CQ_API_KEY from environment, or pass explicitly:
client = cq.connect(url="http://localhost:8002", api_key="lpyIERNvwLqz3rd8gVxITgFp66qqt3RwSFxMd7fCQvc")
#client = cq.connect()
print(f"Cascade Quantum SDK v{cq.__version__}")
print(f"Server: {client.url}")
print(f"Status: {client.status()}")

Cascade Quantum SDK v0.1.0
Server: http://localhost:8002
Status: {'ok': True}


## 1. List Available Datasets

In [2]:
datasets = client.datasets.list()
print(f"Available datasets ({len(datasets)}):")
for ds in datasets:
    print(f"  {ds['id']:20s} {ds['name']:30s} {ds.get('rows', '?')} rows, {ds.get('classes', '?')} classes")

Available datasets (8):
  clinical_trial_v1    Clinical Trial V1 (Synthetic)  ? rows, 2 classes
  diabetes             Pima Diabetes                  768 rows, 2 classes
  iris                 Iris                           150 rows, 3 classes
  mnist_784            MNIST Full (70k)               70000 rows, 4 classes
  mnist_small          MNIST Small (400 samples)      400 rows, 4 classes
  random_gene_expression Random Gene Expression  (Synthetic) ? rows, 2 classes
  wdbc                 WDBC (Breast Cancer)           569 rows, 2 classes
  wine                 Wine                           178 rows, 3 classes


In [3]:
# Select a dataset for this demo
dataset = "wdbc"
print(f"Selected: {dataset}")

Selected: wdbc


## 2. PCA Analysis

Principal Component Analysis reveals the intrinsic dimensionality of the data — how many independent dimensions carry real signal.

In [4]:
pca = client.analysis.pca(dataset, verbose=True)

print(f"\nDataset: {pca['dataset_name']}")
print(f"Features: {pca['features']}")
print(f"Samples: {pca['samples']}")
print(f"Components for 90% variance: {pca['thresholds']['90']}")
print(f"Components for 95% variance: {pca['thresholds']['95']}")
print(f"Elapsed: {pca.get('elapsed_seconds', '?')}s")

  [100%] Done.                                        

Dataset: WDBC (Breast Cancer)
Features: 30
Samples: 569
Components for 90% variance: 7
Components for 95% variance: 10
Elapsed: 0.0s


## 3. Mutual Information Analysis

MI ranks features by how much they predict the class label. The quantum model will use the top-ranked features.

In [5]:
mi = client.analysis.mi(dataset, verbose=True)

print(f"\nTotal MI: {mi['total_mi']:.4f}")
print(f"Features for 90% MI: {mi['thresholds']['90']}")
print(f"\nTop 10 features:")
for f in mi['ranked_features'][:10]:
    print(f"  {f['rank']+1:2d}. {f['feature_name']:25s} MI={f['mi_score']:.4f} ({f['mi_percent']:.1f}%)")

  [100%] Done.                                        

Total MI: 1.3392
Features for 90% MI: 17

Top 10 features:
   1. PC1                       MI=0.4440 (33.2%)
   2. PC2                       MI=0.0867 (6.5%)
   3. PC28                      MI=0.0807 (6.0%)
   4. PC24                      MI=0.0633 (4.7%)
   5. PC17                      MI=0.0584 (4.4%)
   6. PC13                      MI=0.0567 (4.2%)
   7. PC3                       MI=0.0549 (4.1%)
   8. PC25                      MI=0.0496 (3.7%)
   9. PC26                      MI=0.0474 (3.5%)
  10. PC29                      MI=0.0451 (3.4%)


## 4. Quantum Resource Estimation

Estimates the number of qubits needed for the quantum model to achieve high accuracy on this dataset.

In [6]:
qre = client.analysis.qre(dataset, verbose=True)

print(f"\nEstimated qubits: {qre['estimatedQubits']} +/- {qre['standardDeviation']:.1f}")
print(f"Samples: {qre['samples']}, Features: {qre['features']}, Classes: {qre['classes']}")
print(f"Elapsed: {qre.get('elapsed_seconds', '?')}s")

  [100%] Done.                                        

Estimated qubits: 16 +/- 1.4
Samples: 569, Features: 30, Classes: 2
Elapsed: 0.5s


## 5. Model Training

Train a quantum machine learning model. Settings can be overridden — here we use 4 qubits with 1 layer as a quick demo.

In [7]:
result = client.training.run(dataset, verbose=True, settings={
    "num_qubits": 4,
    "min_layers": 1,
    "max_layers": 1,
})

train_id = result['train_id']
print(f"\nModel ID: {train_id}")
print(f"Elapsed: {result.get('elapsed_seconds', '?')}s")


Model ID: 0b7df2f3-8187-4d45-976c-3f78d8cdf53c
Elapsed: ?s


## 6. Inference (Classification)

Test the trained model on held-out test data.

In [8]:
inference = client.inference.run(train_id, verbose=True)

print(f"\nOverall accuracy: {inference['overall_accuracy']:.1%}")
print(f"Overall loss: {inference['overall_loss']:.4f}")
print(f"Test samples: {inference['test_samples']}")
print(f"\nPer-class results:")
for cls in inference['per_class']:
    print(f"  Class {cls['class']}: accuracy={cls['accuracy']:.1%}, samples={cls['samples']}")

  [100%] Done.                                        

Overall accuracy: 75.4%
Overall loss: 0.2994
Test samples: 114

Per-class results:
  Class 1: accuracy=63.9%, samples=72
  Class 2: accuracy=95.2%, samples=42


## 7. Explainable AI

Identify which input features drive the model's classification decisions.

In [9]:
explanation = client.explainable.run(train_id, verbose=True)

print(f"\nDataset: {explanation['dataset_name']}")
print(f"Qubits: {explanation['num_qubits']} ({explanation['num_class_qubits']} class, {explanation['num_qubits'] - explanation['num_class_qubits']} data)")

if explanation.get('qubit_map'):
    print(f"\nKey features (data qubit -> feature):")
    for q in explanation['qubit_map']:
        print(f"  Qubit {q['qubit']}: {q['feature_name']} (col {q['feature_index']}, importance {q['importance']*100:.1f}%)")

print(f"\nConvergence:")
for qid, val in explanation.get('convergence', {}).items():
    print(f"  Class qubit {qid}: {val:.4f}")

  [100%] Done.                                        

Dataset: wdbc
Qubits: 4 (1 class, 3 data)

Key features (data qubit -> feature):
  Qubit 0: V1 (col 0, importance 93.0%)
  Qubit 1: V3 (col 2, importance 0.2%)
  Qubit 2: V20 (col 19, importance 0.8%)

Convergence:
  Class qubit 0: 0.7215


## Summary

This notebook demonstrated the complete Cascade Quantum pipeline:
1. **PCA** revealed the dataset's intrinsic dimensionality
2. **MI** identified the most predictive features
3. **QRE** estimated the qubits needed
4. **Training** built a quantum model
5. **Inference** tested it on held-out data
6. **Explainable AI** showed which features drive decisions

All computation ran on the server. This notebook required only `pip install cascade-quantum`.